# TP 1 : Preprocessing et Pipelines

**Objectif :**
Apprendre à préparer les données et automatiser le workflow ML avec les pipelines scikit-learn.
3 étapes clés :

Prétraitement manuel → encoder + normaliser les données correctement
Pipeline simple → automatiser ces étapes en une seule commande
Pipeline avancée → gérer colonnes mixtes, sélection de features et modèle dans une seule pipeline

---
###  Importation des Bibliothèques

In [36]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures, MinMaxScaler, OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split
import seaborn as sns

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


## 1. Génération d'un Dataset Synthétique

---
####  Chargement du Dataset Diamonds

On utilise le dataset **Diamonds** disponible via `seaborn`. Il contient ~54 000 observations avec les variables :
- **Numériques** : carat, depth, table, price, x, y, z
- **Catégorielles** : cut (coupe), color (couleur), clarity (clarté)

In [37]:
# Charger le dataset depuis Seaborn
df = sns.load_dataset("diamonds")

# Afficher les 5 premières lignes
df.head()

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


## 2. Preprocessing des Variables Numériques

### Train Test Split

####  Séparation Train / Test

On divise le dataset en deux parties :
- **80%** → `train_set` : sert à ajuster (fit) les transformateurs et le modèle
- **20%** → `test_set` : sert à évaluer les performances (jamais vu pendant l'entraînement)

> `random_state=0` assure la reproductibilité des résultats.

In [38]:
train_set, test_set = train_test_split(df,test_size=0.2, random_state=0)

####  Vérification des dimensions

On vérifie que la séparation a bien été effectuée et que les proportions 80/20 sont respectées.

In [39]:
print("train_set :",train_set.shape)
print("test_set :",test_set.shape)

train_set : (43152, 10)
test_set : (10788, 10)


### Encoding
Commencez par explorer votre jeu de données et identifiez les variables catégoriques.<br>
Choisissez ensuite entre OrdinalEncoder et OneHotEncoder pour encoder ces variables : <br>
    - Si les catégories suivent un ordre naturel (ex. : "petit", "moyen", "grand"), utilisez OrdinalEncoder.<br>
    - Si elles n'ont pas d'ordre particulier (ex. : "rouge", "bleu", "vert"), utilisez OneHotEncoder.<br>
    - Faites une recherche sur Internet sur la classification des diamants afin de déterminer la méthode d'encodage la plus appropriée.

---
####  Exploration des Variables Catégorielles

Avant d'encoder, on explore les **valeurs uniques** de chaque colonne catégorielle (`cut`, `color`, `clarity`) pour décider quelle méthode utiliser :
- Si les catégories ont un **ordre naturel** → `OrdinalEncoder`
- Si elles sont **nominales** (sans ordre) → `OneHotEncoder`

Après recherche sur la classification des diamants, les 3 variables ont un ordre → on utilise `OrdinalEncoder`.

In [40]:
# Déterminer les différentes valeurs qui se cachent dans la première colonne catégorielle
valeurs_cut = df['cut'].unique()
print(valeurs_cut)

['Ideal', 'Premium', 'Good', 'Very Good', 'Fair']
Categories (5, str): ['Ideal', 'Premium', 'Very Good', 'Good', 'Fair']


In [41]:
# Déterminer les différentes valeurs qui se cachent dans la deuxième colonne catégorielle
valeurs_color = df['color'].unique()
print(valeurs_color)

['E', 'I', 'J', 'H', 'F', 'G', 'D']
Categories (7, str): ['D', 'E', 'F', 'G', 'H', 'I', 'J']


In [42]:
# Déterminer les différentes valeurs qui se cachent dans  la troisième colonne catégorielle
valeurs_clarity = df['clarity'].unique()
print(valeurs_clarity)

['SI2', 'SI1', 'VS1', 'VS2', 'VVS2', 'VVS1', 'I1', 'IF']
Categories (8, str): ['IF', 'VVS1', 'VVS2', 'VS1', 'VS2', 'SI1', 'SI2', 'I1']


####  Application de l'OrdinalEncoder

On définit l'ordre croissant de chaque catégorie (du moins bon au meilleur) :
- **cut** : `Fair < Good < Very Good < Premium < Ideal`
- **color** : `D` (meilleure transparence) → `J` (moins bonne)
- **clarity** : `IF` (parfaite) → `I1` (inclusions visibles)

L'encodeur transforme ces chaînes de caractères en entiers ordonnés.

In [43]:
#A l'aide d'une petite recherche sur internet, spécifiez l'ordre de chaque catégorie
cut_order = ['Fair','Good', 'Very Good', 'Premium', 'Ideal']
color_order = ['D', 'E', 'F', 'G', 'H', 'I', 'J']
clarity_order = ['IF', 'VVS1', 'VVS2', 'VS1', 'VS2', 'SI1', 'SI2', 'I1']
#Introduire cut_order color_order dans l'Encoder
encoder = OrdinalEncoder(categories=[cut_order, color_order, clarity_order])
encoding_results = encoder.fit_transform(train_set[['cut','color','clarity']])
encoding_results

array([[4., 3., 3.],
       [4., 3., 4.],
       [4., 1., 2.],
       ...,
       [3., 5., 3.],
       [4., 3., 0.],
       [4., 2., 6.]], shape=(43152, 3))

In [44]:
train_set

,carat,cut,color,clarity,depth,table,price,x,y,z
26250,1.63,Ideal,G,VS1,61.7,55.0,15697,7.56,7.60,4.68
31510,0.34,Ideal,G,VS2,62.2,57.0,765,4.47,4.44,2.77
40698,0.40,Ideal,E,VVS2,61.7,56.0,1158,4.73,4.77,2.93
42634,0.58,Premium,H,SI1,62.1,55.0,1332,5.38,5.35,3.33
47714,0.63,Very Good,D,SI1,62.8,57.0,1885,5.40,5.46,3.41
...,...,...,...,...,...,...,...,...,...,...
45891,0.52,Premium,F,VS2,60.7,59.0,1720,5.18,5.14,3.13
52416,0.70,Good,D,SI1,63.6,60.0,2512,5.59,5.51,3.51
42613,0.32,Premium,I,VS1,61.3,58.0,505,4.35,4.39,2.68
43567,0.41,Ideal,G,IF,61.0,57.0,1431,4.81,4.79,2.93


In [45]:
#injecter le résultat de l'encodage dans le tableau de trainset

train_set[['cut','color','clarity']]=encoding_results

train_set

,carat,cut,color,clarity,depth,table,price,x,y,z
26250,1.63,4.0,3.0,3.0,61.7,55.0,15697,7.56,7.60,4.68
31510,0.34,4.0,3.0,4.0,62.2,57.0,765,4.47,4.44,2.77
40698,0.40,4.0,1.0,2.0,61.7,56.0,1158,4.73,4.77,2.93
42634,0.58,3.0,4.0,5.0,62.1,55.0,1332,5.38,5.35,3.33
47714,0.63,2.0,0.0,5.0,62.8,57.0,1885,5.40,5.46,3.41
...,...,...,...,...,...,...,...,...,...,...
45891,0.52,3.0,2.0,4.0,60.7,59.0,1720,5.18,5.14,3.13
52416,0.70,1.0,0.0,5.0,63.6,60.0,2512,5.59,5.51,3.51
42613,0.32,3.0,5.0,3.0,61.3,58.0,505,4.35,4.39,2.68
43567,0.41,4.0,3.0,0.0,61.0,57.0,1431,4.81,4.79,2.93


## Normalisation

---
###  Normalisation avec MinMaxScaler

Après l'encodage, on normalise toutes les variables dans l'intervalle **[0, 1]** grâce à `MinMaxScaler`.

**Formule** : `x_norm = (x - x_min) / (x_max - x_min)`

>  On applique `.fit_transform()` **uniquement sur le train set** pour ne pas laisser fuiter d'information du test set.

In [46]:
scaler = MinMaxScaler()
trani_set = scaler.fit_transform(train_set)
train_set

,carat,cut,color,clarity,depth,table,price,x,y,z
26250,1.63,4.0,3.0,3.0,61.7,55.0,15697,7.56,7.60,4.68
31510,0.34,4.0,3.0,4.0,62.2,57.0,765,4.47,4.44,2.77
40698,0.40,4.0,1.0,2.0,61.7,56.0,1158,4.73,4.77,2.93
42634,0.58,3.0,4.0,5.0,62.1,55.0,1332,5.38,5.35,3.33
47714,0.63,2.0,0.0,5.0,62.8,57.0,1885,5.40,5.46,3.41
...,...,...,...,...,...,...,...,...,...,...
45891,0.52,3.0,2.0,4.0,60.7,59.0,1720,5.18,5.14,3.13
52416,0.70,1.0,0.0,5.0,63.6,60.0,2512,5.59,5.51,3.51
42613,0.32,3.0,5.0,3.0,61.3,58.0,505,4.35,4.39,2.68
43567,0.41,4.0,3.0,0.0,61.0,57.0,1431,4.81,4.79,2.93


### Transformation du test set

---
###  Transformation du Test Set

On applique les **mêmes paramètres** appris sur le train set au test set :
1. `encoder.transform()` → encodage avec les ordres déjà appris
2. `scaler.transform()` → normalisation avec min/max du train set

>  **Important** : on utilise `.transform()` (et non `.fit_transform()`) sur le test set pour éviter la **fuite de données (data leakage)**.

In [47]:
encoding_results = encoder.transform(test_set[['cut','color','clarity']])

In [48]:
#injecter le résultat de l'encodage dans le tableau de testset
test_set[['cut','color','clarity']] = encoding_results
test_set

,carat,cut,color,clarity,depth,table,price,x,y,z
10176,1.10,4.0,4.0,6.0,62.0,55.0,4733,6.61,6.65,4.11
16083,1.29,4.0,4.0,5.0,62.6,56.0,6424,6.96,6.93,4.35
13420,1.20,3.0,5.0,5.0,61.1,58.0,5510,6.88,6.80,4.18
20407,1.50,4.0,2.0,5.0,60.9,56.0,8770,7.43,7.36,4.50
8909,0.90,2.0,2.0,4.0,61.7,57.0,4493,6.17,6.21,3.82
...,...,...,...,...,...,...,...,...,...,...
42208,0.52,1.0,4.0,4.0,63.6,57.0,1289,5.05,5.10,3.23
3638,0.91,2.0,3.0,6.0,60.4,61.0,3435,6.21,6.28,3.77
5508,1.08,2.0,2.0,6.0,63.4,55.0,3847,6.53,6.50,4.13
19535,1.02,4.0,3.0,1.0,61.5,57.0,8168,6.44,6.47,3.97


In [49]:
test_set = scaler.transform(test_set)
test_set

array([[0.18711019, 1.        , 0.66666667, ..., 0.61545624, 0.11290323,
        0.12924528],
       [0.22661123, 1.        , 0.66666667, ..., 0.64804469, 0.11765705,
        0.13679245],
       [0.20790021, 0.75      , 0.83333333, ..., 0.6405959 , 0.11544992,
        0.13144654],
       ...,
       [0.18295218, 0.5       , 0.33333333, ..., 0.60800745, 0.11035654,
        0.12987421],
       [0.17047817, 1.        , 0.5       , ..., 0.59962756, 0.1098472 ,
        0.12484277],
       [0.06237006, 0.25      , 0.16666667, ..., 0.46275605, 0.08488964,
        0.10157233]], shape=(10788, 10))

## Simplifier le tout avec une Pipeline

---
##  Pipelines Scikit-learn

L'objet `Pipeline` enchaîne automatiquement plusieurs étapes de prétraitement (et éventuellement un modèle) dans un seul objet cohérent.

**Avantages :**
- Élimine les risques de **fuite de données**
- Rend le code plus **lisible et maintenable**
- Facilite le **déploiement** en production


In [50]:
from sklearn.pipeline import Pipeline, make_pipeline

In [51]:
df = sns.load_dataset("diamonds")
df.head()

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [52]:
train_set, test_set = train_test_split(df,test_size=0.2,random_state=0)

####  Pipeline Simple : Encoder → Scaler

Première pipeline de base : `OrdinalEncoder` + `MinMaxScaler`.
Un seul appel à `.fit_transform()` exécute les deux étapes dans l'ordre.

In [53]:
pipeline = Pipeline(steps=[("Encoder",OrdinalEncoder()),
                           ("Normaliser",MinMaxScaler())])

In [54]:
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('Encoder', ...), ('Normaliser', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute.",'auto'
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'use_encoded_value'}, default='error'When set to 'error' an error will be raised in case an unknowncategorical feature is present during transform. When set to'use_encoded_value', the encoded value of unknown categories will beset to the value given for the parameter `unknown_value`. In:meth:`inverse_transform`, an unknown category will be denoted as None... versionadded:: 0.24",'error'
,"unknown_value unknown_value: int or np.nan, default=NoneWhen the parameter handle_unknown is set to 'use_encoded_value', thisparameter is required and will set the encoded value of unknowncategories. It has to be distinct from the values used to encode any ofthe categories in `fit`. If set to np.nan, the `dtype` parameter mustbe a float dtype... versionadded:: 0.24",None
,"encoded_missing_value encoded_missing_value: int or np.nan, default=np.nanEncoded value of missing categories. If set to `np.nan`, then the `dtype`parameter must be a float dtype... versionadded:: 1.1",nan
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.3 Read more in the :ref:`User Guide `.",None
,"max_categories max_categories: int, default=NoneSpecifies an upper limit to the number of output categories for each inputfeature when considering infrequent categories. If there are infrequentcategories, `max_cat

In [55]:
pipeline.fit_transform(train_set)

array([[0.53358209, 0.5       , 0.5       , ..., 0.69652651, 0.70879121,
        0.67486339],
       [0.05223881, 0.5       , 0.5       , ..., 0.13162706, 0.13003663,
        0.15300546],
       [0.07462687, 0.5       , 0.16666667, ..., 0.17915905, 0.19047619,
        0.19672131],
       ...,
       [0.04477612, 0.75      , 0.83333333, ..., 0.10968921, 0.12087912,
        0.1284153 ],
       [0.07835821, 0.5       , 0.5       , ..., 0.19378428, 0.19413919,
        0.19672131],
       [0.26492537, 0.5       , 0.33333333, ..., 0.45521024, 0.45054945,
        0.43442623]], shape=(43152, 10))

### Pipeline composée

<img src="pipeline.png" alt="Texte alternatif" width="450" height="500">


---
###  Pipeline Composée avec ColumnTransformer

`ColumnTransformer` permet d'appliquer des **transformations différentes** à des colonnes différentes :
- Colonnes catégorielles → `OrdinalEncoder` avec ordres définis
- Colonnes numériques → `passthrough` (inchangées)

On intègre ensuite ce `ColumnTransformer` comme première étape d'une `Pipeline` pour ajouter la normalisation.

In [56]:
from sklearn.compose import ColumnTransformer 

In [57]:
# définir une liste des variables catégorielles 
categorical_cols = ["cut","color","clarity"]

In [58]:
column_transformer = ColumnTransformer(transformers=[("Encoder",OrdinalEncoder(categories=[cut_order, color_order, clarity_order]),categorical_cols)],
                                       remainder="passthrough")
#Intégrer column_transformer dans une pipeline
pipeline = Pipeline(steps=[("Encoder",column_transformer),
                           ("Scaler",MinMaxScaler())
                          ])
pipeline.fit_transform(train_set)

array([[1.        , 0.5       , 0.42857143, ..., 0.70391061, 0.12903226,
        0.14716981],
       [1.        , 0.5       , 0.57142857, ..., 0.41620112, 0.075382  ,
        0.08710692],
       [1.        , 0.16666667, 0.28571429, ..., 0.44040968, 0.08098472,
        0.09213836],
       ...,
       [0.75      , 0.83333333, 0.42857143, ..., 0.40502793, 0.07453311,
        0.08427673],
       [1.        , 0.5       , 0.        , ..., 0.44785847, 0.08132428,
        0.09213836],
       [1.        , 0.33333333, 0.85714286, ..., 0.58100559, 0.10509338,
        0.11949686]], shape=(43152, 10))

In [59]:
pipeline.transform(test_set)

array([[1.        , 0.66666667, 0.85714286, ..., 0.61545624, 0.11290323,
        0.12924528],
       [1.        , 0.66666667, 0.71428571, ..., 0.64804469, 0.11765705,
        0.13679245],
       [0.75      , 0.83333333, 0.71428571, ..., 0.6405959 , 0.11544992,
        0.13144654],
       ...,
       [0.5       , 0.33333333, 0.85714286, ..., 0.60800745, 0.11035654,
        0.12987421],
       [1.        , 0.5       , 0.14285714, ..., 0.59962756, 0.1098472 ,
        0.12484277],
       [0.25      , 0.16666667, 0.28571429, ..., 0.46275605, 0.08488964,
        0.10157233]], shape=(10788, 10))

### Exercice : Création des pipelines
<img src="./pipelineImage/pipe1.png" width="200">

---

#### Pipeline Niveau 1 — Feature Engineering + Sélection + Normalisation

Cette pipeline enchaîne trois étapes :
1. **`PolynomialFeatures(degree=2)`** : génère des features polynomiales (termes croisés et carrés) — enrichit le jeu de données
2. **`SelectKBest(f_regression, k=5)`** : conserve les 5 meilleures features selon leur corrélation avec la cible
3. **`MinMaxScaler`** : normalise les features sélectionnées dans [0,1]

>  Très utile pour la régression sur des données peu nombreuses.

In [60]:
from sklearn.feature_selection import SelectKBest, f_regression, f_classif
pipeNiveau1 = Pipeline(steps=[
    ("FeatureEngineering",PolynomialFeatures(degree = 2, include_bias=True)),
    ("SelectFeatures", SelectKBest(score_func=f_regression, k=5)),
    ("Normalisation",MinMaxScaler())])
pipeNiveau1

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('FeatureEngineering', ...), ('SelectFeatures', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"degree degree: int or tuple (min_degree, max_degree), default=2If a single int is given, it specifies the maximal degree of thepolynomial features. If a tuple `(min_degree, max_degree)` is passed,then `min_degree` is the minimum and `max_degree` is the maximumpolynomial degree of the generated features. Note that `min_degree=0`and `min_degree=1` are equivalent as outputting the degree zero term isdetermined by `include_bias`.",2
,"interaction_only interaction_only: bool, default=FalseIf `True`, only interaction features are produced: features that areproducts of at most `degree` *distinct* input features, i.e. terms withpower of 2 or higher of the same input feature are excluded:- included: `x[0]`, `x[1]`, `x[0] * x[1]`, etc.- excluded: `x[0] ** 2`, `x[0] ** 2 * x[1]`, etc.",False
,"include_bias include_bias: bool, default=TrueIf `True` (default), then include a bias column, the feature in whichall polynomial powers are zero (i.e. a column of ones - acts as anintercept term in a linear model).",True
,"order order: {'C', 'F'}, default='C'Order of output array in the dense case. `'F'` order is faster tocompute, but may slow down subsequent estimators... versionadded:: 0.21",'C'
,"score_func score_func: callable, default=f_classifFunction taking two arrays X and y, and returning a pair of arrays(scores, pvalues) or a single array with scores.Default is f_classif (see below ""See Also""). The default function onlyworks with classification tasks... versionadded:: 0.18",<function f_r...001A078E88360>
,"k k: int or ""all"", default=10Number of top features to select.The ""all"" option bypasses selection, for use in a parameter search.",5
,"feature_range feature_range: tuple (min, max), default=(0, 1)Desired range of transformed data.","(0, ...)"


In [61]:
# Création d’un jeu de données de régression pour tester la pipeline
from sklearn.datasets import make_regression
X, y =make_regression (n_samples = 100, n_features = 4, noise = 0.1, random_state = 42)
pipeNiveau1.fit_transform(X,y)

array([[6.03262586e-01, 3.69231960e-01, 0.00000000e+00, 2.56728701e-01,
        2.33507240e-03],
       [3.17470504e-01, 2.30020341e-01, 5.31391733e-01, 2.82829305e-01,
        2.61159206e-02],
       [1.79406459e-01, 3.63601033e-01, 2.62980978e-01, 1.97733130e-01,
        1.58498975e-03],
       [5.01996265e-01, 3.56237034e-01, 3.41204541e-01, 2.36055325e-01,
        8.21851895e-04],
       [1.68098199e-01, 3.05211385e-01, 3.56230227e-01, 2.67861072e-01,
        2.31513390e-03],
       [6.06092365e-01, 3.17413318e-01, 2.84027264e-01, 2.11196626e-01,
        8.79832415e-04],
       [3.72361791e-01, 3.82066750e-01, 5.34877701e-01, 2.17912274e-01,
        4.58419083e-03],
       [5.31328384e-01, 1.75804281e-01, 1.49194213e-01, 1.43762821e-01,
        5.92457930e-02],
       [1.18406659e-01, 1.22477246e-01, 4.08446003e-01, 5.43306317e-01,
        1.04885300e-01],
       [2.59194826e-01, 5.02191890e-01, 8.01024303e-01, 9.89395779e-02,
        6.19867159e-02],
       [2.50496227e-01, 3.6869

---
#### Pipeline Niveau 2 — Preprocessing Mixte (Numérique + Catégoriel)

On génère un dataset mixte :
- 2 colonnes **numériques continues** (`X1`, `X2`)
- 1 colonne **catégorielle nominale** (`Ville` : Paris, Marseille, Belfort)

Le `ColumnTransformer` applique :
- `MinMaxScaler` sur les colonnes numériques
- `OneHotEncoder` sur la colonne catégorielle (pas d'ordre → on ne peut pas utiliser `OrdinalEncoder`)

In [62]:
#Création d’un jeu de données de régression pour tester la deuxième pipeline
np.random.seed(42)
n_samples = 100
X_numeric = np.random.rand(n_samples, 2) # 2 variables numériques
X_categorical = np.random.choice(['Paris','Marseille','Belfort'],size=(n_samples, 1))
df = pd.DataFrame(np.hstack((X_numeric,X_categorical)),columns=["X1","X2","Ville"])
df

,X1,X2,Ville
0,0.3745401188473625,0.9507143064099162,Belfort
1,0.7319939418114051,0.5986584841970366,Paris
2,0.15601864044243652,0.15599452033620265,Marseille
3,0.05808361216819946,0.8661761457749352,Marseille
4,0.6011150117432088,0.7080725777960455,Belfort
...,...,...,...
95,0.09310276780589921,0.8972157579533268,Marseille
96,0.9004180571633305,0.6331014572732679,Belfort
97,0.3390297910487007,0.3492095746126609,Paris
98,0.7259556788702394,0.8971102599525771,Marseille


<img src="./pipelineImage/pipe2.png" width="300">

In [63]:
# Colonnes par type
num_cols = ["X1", "X2"]
cat_cols = ["Ville"]

# Assurer le bon type numérique (car df a été construit avec np.hstack)
df[num_cols] = df[num_cols].astype(float)

# Pipeline niveau 2 : prétraitement mixte (numérique + catégoriel)
pipeNiveau2 = Pipeline(steps=[
    ("Preprocessing", ColumnTransformer(
        transformers=[
            ("num", MinMaxScaler(), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
        ],
        remainder="drop"
    ))
])

pipeNiveau2

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('Preprocessing', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse m

In [64]:
pipeNiveau2.fit_transform(df).shape

(100, 5)

<img src="./pipelineImage/pipe3.png" width="350">

---
#### Pipeline Niveau 3 — Preprocessing Mixte + Sélection de Features

On ajoute une étape de **sélection de variables** :
1. `ColumnTransformer` : normalise les numériques + encode les catégorielles
2. `SelectKBest(f_regression, k=5)` : ne garde que les 5 features les plus informatives

>  Réduit la dimensionnalité et améliore la généralisation du modèle.

In [65]:
np.random.seed(42)
n_samples = 100
X_numeric = np.random.rand(n_samples, 2) # 2 variables numérique
X_categorical = np.random.choice(['A','B','C','D'],size = (n_samples,2)) # 2 variables catégorielles
y=np.random.rand(n_samples)*10 # variable cible
#création d'un DataFrame
df=pd.DataFrame(np.hstack((X_numeric,X_categorical)), columns=["X1","X2","Cat 1","Cat 2"])
df["y"]= y
#Définition des colonnes catégorielles
cat_cols = ["Cat 1","Cat 2"]
df['Cat 2']

0     D
1     A
2     D
3     D
4     B
     ..
95    D
96    D
97    D
98    C
99    B
Name: Cat 2, Length: 100, dtype: str

In [66]:
# Colonnes
num_cols = ["X1", "X2"]
cat_cols = ["Cat 1", "Cat 2"]

# Assurer le bon type des colonnes numériques
df[num_cols] = df[num_cols].astype(float)

# Séparer les variables explicatives et la cible
X = df.drop(columns=["y"])

y = df["y"]

# Pipeline de preprocessing + sélection de variables
pipelineNiveau3 = Pipeline(steps=[
    ("Preprocessing", ColumnTransformer(
        transformers=[
            ("num", MinMaxScaler(), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
        ],
        remainder="drop"
    )),
    ("SelectFeatures", SelectKBest(score_func=f_regression, k=5))
])

# Affichage de la pipeline
pipelineNiveau3

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('Preprocessing', ...), ('SelectFeatures', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different trans

In [67]:
# Appliquer la pipeline niveau 3 sur X et y
X_niv3 = pipelineNiveau3.fit_transform(X, y)
print("Shape après pipelineNiveau3:", X_niv3.shape)
pd.DataFrame(X_niv3).head()

Shape après pipelineNiveau3: (100, 5)


,0,1,2,3,4
0,0.0,0.0,1.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,1.0,0.0,0.0
3,0.0,1.0,0.0,0.0,0.0
4,0.0,0.0,1.0,1.0,0.0


<img src="./pipelineImage/pipe4.png" width="350">

---
#### Pipeline Niveau 4 — Pipeline Complète avec Modèle

La pipeline la plus complète intègre **tout le workflow ML** :
1. **`ColumnTransformer`** : prétraitement des features
2. **`SelectKBest`** : sélection des 5 meilleures features
3. **`LinearRegression`** : modèle de régression linéaire

Un seul `.fit()` entraîne toute la chaîne. Un seul `.predict()` applique tout le prétraitement + la prédiction.

>  **Bonne pratique en ML** : toujours intégrer le modèle dans la pipeline pour garantir l'absence de fuite de données et faciliter le déploiement.

In [68]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# Pipeline complète : preprocessing + sélection + modèle
pipelineNiveau4 = Pipeline(steps=[
    ("Preprocessing", ColumnTransformer(
        transformers=[
            ("num", MinMaxScaler(), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
        ],
        remainder="drop"
    )),
    ("SelectFeatures", SelectKBest(score_func=f_regression, k=5)),
    ("Model", LinearRegression())
])

# Entraînement et évaluation
pipelineNiveau4.fit(X, y)
y_pred = pipelineNiveau4.predict(X)

print("R2:", round(r2_score(y, y_pred), 4))
print("MSE:", round(mean_squared_error(y, y_pred), 4))
pd.DataFrame({"y_reel": y.head(10), "y_pred": y_pred[:10]})

R2: 0.0893
MSE: 7.7636


,y_reel,y_pred
0,0.516817,3.996801
1,5.313546,5.035328
2,5.406351,3.996801
3,6.374299,6.306142
4,7.260913,4.211833
5,9.758521,5.035328
6,5.163003,6.306142
7,3.229565,5.035328
8,7.951862,5.421051
9,2.708323,4.211833


---
## Conclusion du TP 1

Ce TP nous a permis de maîtriser les étapes essentielles du **preprocessing** et de la construction de **pipelines scikit-learn** :

| Concept | Outil | Rôle |
|---|---|---|
| Séparation données | `train_test_split` | Éviter le data leakage |
| Encodage ordonné | `OrdinalEncoder` | Variables catégorielles ordonnées |
| Encodage nominal | `OneHotEncoder` | Variables sans ordre naturel |
| Normalisation | `MinMaxScaler` | Ramener dans [0,1] |
| Pipeline simple | `Pipeline` | Enchaîner des transformations |
| Pipeline composée | `ColumnTransformer` | Transformations par colonnes |
| Sélection features | `SelectKBest` | Réduire la dimensionnalité |
| Pipeline complète | `Pipeline` + modèle | Workflow ML complet |

>  La `Pipeline` est l'outil fondamental en ML pour un code propre, reproductible et sans fuite de données.
